## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [1]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [2]:
# imports for langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter

In [3]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [5]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase
# Thank you Mark D. and Zoya H. for fixing a bug here..

folders = glob.glob("knowledge-base/*")

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [6]:
len(documents)

31

In [11]:
documents[15]

Document(metadata={'source': 'knowledge-base\\employees\\Alex Chen.md', 'doc_type': 'employees'}, page_content='# HR Record\n\n# Alex Chen\n\n## Summary\n- **Date of Birth:** March 15, 1990  \n- **Job Title:** Backend Software Engineer  \n- **Location:** San Francisco, California  \n\n## Insurellm Career Progression\n- **April 2020:** Joined Insurellm as a Junior Backend Developer. Focused on building APIs to enhance customer data security.\n- **October 2021:** Promoted to Backend Software Engineer. Took on leadership for a key project developing a microservices architecture to support the company\'s growing platform.\n- **March 2023:** Awarded the title of Senior Backend Software Engineer due to exemplary performance in scaling backend services, reducing downtime by 30% over six months.\n\n## Annual Performance History\n- **2020:**  \n  - Completed onboarding successfully.  \n  - Met expectations in delivering project milestones.  \n  - Received positive feedback from the team leads.\

In [12]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1088, which is longer than the specified 1000


In [13]:
len(chunks)

123

In [33]:
chunks[80]

Document(metadata={'source': 'knowledge-base\\employees\\Jordan Blake.md', 'doc_type': 'employees'}, page_content="## Annual Performance History\n- **2021:** First year at Insurellm; achieved 90% of monthly targets.  \n  - **Feedback:** Strong potential shown in lead generation; needs improvement in follow-up techniques.  \n- **2022:** Achieved 120% of targets; pioneered outreach strategies that increased customer engagement.  \n  - **Feedback:** Jordan's innovative approach contributed significantly to team success; recommended for leadership training.  \n- **2023:** Set to exceed annual targets by 30% in Q3; initiated successful partnerships that broadened market reach.  \n  - **Feedback:** Exceptional communicator; exemplifies the values of Insurellm and promotes team collaboration.\n\n## Compensation History\n- **2021-06:** Starting Salary: $50,000  \n- **2022-04:** Merit-based increase: $55,000 (based on performance review)  \n- **2023-06:** Performance bonus awarded: $5,000 (for 

In [15]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: company, products, employees, contracts


In [35]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")

page_content='3. **Regular Updates:** Insurellm will offer ongoing updates and enhancements to the Homellm platform, including new features and security improvements.

4. **Feedback Implementation:** Insurellm will actively solicit feedback from GreenValley Insurance to ensure Homellm continues to meet their evolving needs.

---

**Signatures:**

_________________________________  
**[Name]**  
**Title**: CEO  
**Insurellm, Inc.**

_________________________________  
**[Name]**  
**Title**: COO  
**GreenValley Insurance, LLC**  

---

This agreement represents the complete understanding of both parties regarding the use of the Homellm product and supersedes any prior agreements or communications.' metadata={'source': 'knowledge-base\\contracts\\Contract with GreenValley Insurance for Homellm.md', 'doc_type': 'contracts'}
_________
page_content='## Support

1. **Customer Support**: Velocity Auto Solutions will have access to Insurellm’s customer support team via email or chatbot, availa

---
Bu kod, LangChain ile belge işleme sürecini gösteriyor. Adım adım açıklayayım:

## Kod Analizi:

**Text Splitter Oluşturma:**
```python
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
```
Belgeler 1000 karakterlik parçalara bölünüyor, 200 karakter örtüşme ile. Bu örtüşme çok önemli çünkü bilgi kaybını önler.

**Chunk Kontrolü:**
Kod gösteriyor ki 1088 karakterlik bir chunk oluşmuş (1000'den fazla olduğu için uyarı veriyor). Toplam 123 chunk üretilmiş.

**Metadata Görünümü:**
```python
Document(metadata={'source': 'knowledge-base\\employees\\Jordan Blake.md', 'doc_type': 'employees'})
```
Her chunk'ın metadata'sı var: dosya yolu ve belge tipi bilgisi.

**Belge Tipleri:**
Sistem farklı belge tiplerini tanımlıyor: company, products, employees, contracts.

**İçerik Örneği:**
Jordan Blake isimli çalışanın performans değerlendirmesi görünüyor - maaş artışları, hedefler, geri bildirimler içeriyor.

## Alttaki Türkçe Notun Önemi:

Bu not RAG'in temel felsefesini açıklıyor. Geleneksel arama "kelime eşleştirme" yapıyordu. Ama vektör arama **anlamsal arama** yapıyor. 

Örneğin kullanıcı "maaş artışı" aradığında, metinde "salary increase" geçse bile bulabilir. Ya da "performans" aradığında "başarı", "hedef", "değerlendirme" içeren parçaları da getirebilir.

Bu kod tam olarak bunu hazırlıyor: belgeleri anlamsal olarak aranabilir parçalara bölerek, AI'ın sadece kelime eşleştirmesi değil, anlam eşleştirmesi yapmasını sağlıyor. RAG'in gücü de buradan geliyor - kullanıcının aradığı şeyin arkasındaki anlamı kavrayarak daha akıllı sonuçlar döndürüyor.

# ÇOK ÖNEMLİ NOT

**Chunk'lar RAG'in temel yapı taşı.**

## Neden Chunk'lar Bu Kadar Kritik:

**LLM Token Sınırları:** LLM'ler sınırlı token alabilir. 100 sayfalık bir belgeyi tek seferde işleyemez. Chunk'lar bu sorunu çözer.

**Hassas Arama:** Büyük belge yerine, soruyla ilgili spesifik parçaları bulur. "Maaş politikası" sorduğunda, 500 sayfalık şirket el kitabından sadece maaş bölümünü getirir.

**Bağlam Korunması:** Chunk'lar arası örtüşme (overlap) sayesinde bilgi kaybı olmaz. Bir cümle ortada kesilirse, bir sonraki chunk'ta devam eder.

**Verimlilik:** Tüm belge yerine sadece ilgili 2-3 chunk LLM'e gönderilir. Bu hem hızlı hem ucuz.

## RAG Süreci:
1. **Belge → Chunk'lara böl**
2. **Chunk'lar → Vektörlere dönüştür** 
3. **Vektörleri → Vektör DB'de sakla**
4. **Soru geldiğinde → En yakın chunk'ları bul**
5. **Chunk'lar + Soru → LLM'e gönder**

Chunk'lar olmasaydı RAG çalışmazdı. Çünkü:
- Ne arayacağını bilemezdin (çok büyük belgeler)
- LLM'e ne göndereceğini bilemezdin (token sınırı)
- Alakasız bilgiler gürültü yaratırdı

**Özet:** Chunk'lar RAG'in DNA'sı. Doğru chunk stratejisi = Başarılı RAG sistemi!